# Tema 1.1 - CompraFácil  
## Práctica guiada: integración de CRISP-DM y Agile mediante Canvas de proyecto analítico

Este notebook desarrolla el caso de estudio **CompraFácil** mediante una metodología híbrida **CRISP-DM + Agile**.

### Propósito de la práctica
Planificar y ejecutar un proyecto básico de ciencia de datos para predecir el abandono del carrito de compras, manteniendo:
- rigor analítico,
- alineación con los objetivos del negocio,
- y entrega iterativa de valor.

### Productos esperados
Al ejecutar este notebook se deben generar dos archivos:
- `metrics.csv`
- `predictions.csv`

Estos archivos se utilizarán en prácticas posteriores.

In [ ]:
# ============================================================
# 0. Configuración inicial
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

SEED = 42
rng = np.random.default_rng(SEED)

print("Librerías cargadas correctamente.")

## Sprint 0. Comprensión del negocio
En esta fase se define el problema, el objetivo, los datos requeridos y los entregables iniciales del proyecto.

In [ ]:
# ============================================================
# 1. Sprint 0 - Canvas de proyecto analítico
# ============================================================

canvas_analitico = {
    "Caso de estudio": "CompraFácil",
    "Problema de negocio": (
        "Una proporción importante de usuarios agrega productos al carrito, "
        "pero abandona la transacción antes de concretar la compra."
    ),
    "Objetivo de negocio": (
        "Disminuir la tasa de abandono del carrito en un 15% el siguiente trimestre."
    ),
    "Pregunta analítica": (
        "¿Qué variables predicen con mayor precisión el abandono del carrito?"
    ),
    "Datos requeridos": (
        "Historial de compras, tiempos de sesión, método de pago y categoría de producto."
    ),
    "Entregables del sprint": (
        "Conjunto de datos limpio y notebook inicial de exploración."
    ),
    "Stakeholder principal": "Área de marketing",
    "Historia de usuario": (
        "Como gerente de marketing, necesito identificar usuarios con alta "
        "probabilidad de abandono para intervenir con mensajes personalizados."
    ),
    "Restricciones": (
        "Se requiere una solución interpretable, rápida de implementar y alineada "
        "con el objetivo comercial."
    )
}

canvas_df = pd.DataFrame({
    "Elemento": list(canvas_analitico.keys()),
    "Contenido": list(canvas_analitico.values())
})

display(Markdown("### Canvas de proyecto analítico"))
display(canvas_df)

## Sprint 1. Comprensión y preparación de datos
En esta fase se simula un conjunto de datos del caso **CompraFácil**, se realiza una exploración inicial y se preparan los datos para modelado.

In [ ]:
# ============================================================
# 2. Sprint 1 - Generación del dataset simulado
# ============================================================

n = 5000

payment_methods = ["credit_card", "debit_card", "paypal", "cash_on_delivery", "bank_transfer"]
product_categories = ["electronics", "fashion", "home", "beauty", "sports"]
traffic_sources = ["organic", "ads", "social", "email", "direct"]
devices = ["mobile", "desktop", "tablet"]

df = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "session_time_min": rng.gamma(shape=2.5, scale=4.0, size=n).round(2),
    "pages_viewed": rng.integers(2, 25, size=n),
    "items_in_cart": rng.integers(1, 6, size=n),
    "cart_value": rng.normal(1200, 450, size=n).clip(80, 5000).round(2),
    "prior_purchases": rng.poisson(3, size=n),
    "days_since_last_purchase": rng.integers(1, 365, size=n),
    "checkout_delay_sec": rng.normal(95, 40, size=n).clip(5, 400).round(1),
    "payment_method": rng.choice(payment_methods, size=n, p=[0.38, 0.24, 0.18, 0.12, 0.08]),
    "product_category": rng.choice(product_categories, size=n, p=[0.28, 0.24, 0.20, 0.14, 0.14]),
    "traffic_source": rng.choice(traffic_sources, size=n, p=[0.22, 0.27, 0.20, 0.11, 0.20]),
    "device_type": rng.choice(devices, size=n, p=[0.58, 0.32, 0.10]),
    "discount_applied": rng.integers(0, 2, size=n),
    "weekend_session": rng.integers(0, 2, size=n),
    "support_contacts": rng.poisson(0.8, size=n)
})

# Variables derivadas
df["avg_page_time"] = (df["session_time_min"] * 60 / df["pages_viewed"]).round(2)
df["is_high_value_cart"] = (df["cart_value"] >= 1800).astype(int)

# Generación sintética de la variable objetivo
logit = (
    0.70
    + 0.0070 * df["checkout_delay_sec"]
    + 0.0045 * df["days_since_last_purchase"]
    + 0.18 * (df["device_type"] == "mobile").astype(int)
    + 0.22 * (df["payment_method"] == "cash_on_delivery").astype(int)
    + 0.14 * (df["traffic_source"] == "ads").astype(int)
    + 0.11 * (df["traffic_source"] == "social").astype(int)
    + 0.09 * df["support_contacts"]
    - 0.032 * df["session_time_min"]
    - 0.090 * df["prior_purchases"]
    - 0.260 * df["discount_applied"]
    - 0.140 * df["is_high_value_cart"]
    - 0.030 * df["pages_viewed"]
)

prob_abandon = 1 / (1 + np.exp(-logit))
df["abandon_cart"] = rng.binomial(1, np.clip(prob_abandon, 0.02, 0.98), size=n)

# Introducir algunos valores faltantes para simular un caso real
for col, frac in [
    ("session_time_min", 0.03),
    ("payment_method", 0.02),
    ("product_category", 0.02),
    ("checkout_delay_sec", 0.02)
]:
    idx = rng.choice(df.index, size=int(n * frac), replace=False)
    df.loc[idx, col] = np.nan

print("Dataset generado correctamente.")
print("Dimensiones:", df.shape)
display(df.head())

In [ ]:
# ============================================================
# 3. Sprint 1 - Exploración inicial y limpieza
# ============================================================

display(Markdown("### Valores faltantes por variable"))
missing_df = df.isna().sum().sort_values(ascending=False).reset_index()
missing_df.columns = ["variable", "missing_values"]
display(missing_df[missing_df["missing_values"] > 0])

display(Markdown("### Tasa de abandono observada"))
abandon_rate = df["abandon_cart"].mean()
print(f"Tasa de abandono del carrito: {abandon_rate:.2%}")

display(Markdown("### Resumen estadístico"))
display(df.describe(include="all").transpose().head(15))

# Visualización 1: Distribución de la variable objetivo
plt.figure(figsize=(6, 4))
df["abandon_cart"].value_counts().sort_index().plot(kind="bar")
plt.title("Distribución del abandono del carrito")
plt.xticks([0, 1], ["No abandona", "Abandona"], rotation=0)
plt.ylabel("Frecuencia")
plt.show()

# Visualización 2: Tasa de abandono por dispositivo
plt.figure(figsize=(6, 4))
df.groupby("device_type")["abandon_cart"].mean().sort_values(ascending=False).plot(kind="bar")
plt.title("Tasa de abandono por dispositivo")
plt.ylabel("Tasa de abandono")
plt.show()

# En este caso conservamos los registros y resolveremos faltantes en el pipeline
clean_df = df.copy()

print("Preparación inicial completada. Los valores faltantes se resolverán con imputación.")

## Sprint 2-3. Modelado
En esta fase se entrenan y comparan dos modelos base:
- Regresión logística
- Árbol de decisión

Las métricas de evaluación serán:
- Precision
- Recall
- F1 Score
- AUC

In [ ]:
# ============================================================
# 4. Sprint 2-3 - Definición de variables y partición
# ============================================================

target = "abandon_cart"
id_col = "customer_id"

X = clean_df.drop(columns=[target])
y = clean_df[target]

numeric_features = [
    "session_time_min",
    "pages_viewed",
    "items_in_cart",
    "cart_value",
    "prior_purchases",
    "days_since_last_purchase",
    "checkout_delay_sec",
    "discount_applied",
    "weekend_session",
    "support_contacts",
    "avg_page_time",
    "is_high_value_cart"
]

categorical_features = [
    "payment_method",
    "product_category",
    "traffic_source",
    "device_type"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=SEED,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

In [ ]:
# ============================================================
# 5. Sprint 2-3 - Entrenamiento de modelos base
# ============================================================

models = {
    "logistic_regression": LogisticRegression(
        max_iter=1200,
        class_weight="balanced",
        random_state=SEED
    ),
    "decision_tree": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=SEED
    )
}

metrics_records = []
predictions_store = {}

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_prob = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.50).astype(int)

    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)

    metrics_records.append({
        "model": model_name,
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
        "f1_score": round(float(f1), 4),
        "auc": round(float(auc), 4),
        "train_size": int(len(X_train)),
        "test_size": int(len(X_test)),
        "default_threshold": 0.50
    })

    predictions_store[model_name] = pd.DataFrame({
        "customer_id": X_test[id_col].values,
        "y_true": y_test.values,
        "y_proba": y_prob,
        "y_pred_default": y_pred,
        "model": model_name
    })

metrics_df = pd.DataFrame(metrics_records).sort_values(
    by=["auc", "f1_score", "recall", "precision"],
    ascending=False
).reset_index(drop=True)

best_model_name = metrics_df.loc[0, "model"]
metrics_df["selected_model"] = metrics_df["model"].eq(best_model_name)

display(Markdown("### Resultados comparativos"))
display(metrics_df)

print(f"Modelo seleccionado automáticamente: {best_model_name}")

## Sprint 4. Evaluación
En esta fase se validan resultados y se revisan métricas con enfoque de negocio y con stakeholders.

In [ ]:
# ============================================================
# 6. Sprint 4 - Evaluación del mejor modelo
# ============================================================

best_predictions = predictions_store[best_model_name].copy().sort_values("customer_id").reset_index(drop=True)

display(Markdown(f"### Predicciones del modelo seleccionado: {best_model_name}"))
display(best_predictions.head())

print("Reporte de clasificación:")
print(classification_report(
    best_predictions["y_true"],
    best_predictions["y_pred_default"],
    digits=4
))

print("Matriz de confusión:")
print(confusion_matrix(
    best_predictions["y_true"],
    best_predictions["y_pred_default"]
))

## Sprint 5. Implementación
En esta fase se preparan los artefactos iniciales de despliegue y documentación.  
Para esta práctica, los artefactos que se exportarán son:
- `metrics.csv`
- `predictions.csv`

In [ ]:
# ============================================================
# 7. Sprint 5 - Exportación de artefactos
# ============================================================

metrics_path = "metrics.csv"
predictions_path = "predictions.csv"

metrics_df.to_csv(metrics_path, index=False)
best_predictions.to_csv(predictions_path, index=False)

print(f"Archivo generado: {metrics_path}")
print(f"Archivo generado: {predictions_path}")

display(Markdown("### Vista previa de metrics.csv"))
display(pd.read_csv(metrics_path))

display(Markdown("### Vista previa de predictions.csv"))
display(pd.read_csv(predictions_path).head())

In [ ]:
# ============================================================
# 8. Descarga opcional en Google Colab
# ============================================================

try:
    from google.colab import files
    files.download("metrics.csv")
    files.download("predictions.csv")
except Exception:
    print("Si no estás en Google Colab, ignora este mensaje. Los archivos ya fueron creados en el directorio de trabajo.")

## Cierre de la práctica

Con este notebook se completó el flujo base del caso **CompraFácil**:

1. Se definió el problema de negocio.
2. Se elaboró el canvas analítico.
3. Se generó y exploró un dataset simulado.
4. Se entrenaron dos modelos base.
5. Se evaluaron resultados con métricas estándar.
6. Se exportaron los archivos `metrics.csv` y `predictions.csv`.

Estos artefactos quedan listos para las siguientes prácticas.